[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/visualizacion_datos/entregables/corrección/02_analisis_aclaratorio.ipynb)

# Notebook 02 — Análisis Aclaratorio: Transferibilidad a Colombia

**Pregunta central:**  
> *Si queremos construir una herramienta de alerta de deslizamientos en Colombia, ¿cuál de los modelos existentes debería usarse como punto de partida?*

**Dataset base:** Landslide4Sense — modelos entrenados en Nepal, Perú e Italia  
**Colombia:** no tiene dataset etiquetado propio — el análisis evalúa transferibilidad

---

Este notebook NO describe datos. Toma los hallazgos del Notebook 01 y los convierte en **cinco preguntas de negocio** con implicaciones concretas para Colombia:

| # | Pregunta | Implicación |
|---|----------|-------------|
| A1 | ¿Los resultados de la literatura son comparables con los nuestros? | Saber si podemos confiar en los benchmarks publicados |
| A2 | ¿Qué tan grande es la brecha si aplicamos estos modelos en Colombia? | Saber qué tan lejos estamos del estado del arte |
| A3 | ¿Vale la pena invertir en arquitectura compleja para Colombia? | Decidir si se justifica el costo computacional |
| A4 | ¿Qué modelo usar como punto de partida en Colombia? | La decisión final — perfil completo de cada modelo |


In [ ]:
import os, sys, pandas as pd, numpy as np, json

# ── 1. Detectar entorno ─────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── 2. Configurar matplotlib ────────────────────────────────────
import matplotlib
if not IN_COLAB:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# ── 3. Rutas y actualización del repo ───────────────────────────
if IN_COLAB:
    REPO = 'Landslides_-Applied-ML-Course'
    if not os.path.exists(REPO):
        os.system(f'git clone https://github.com/apmontesp/{REPO}')
    else:
        os.system(f'git -C {REPO} pull')   # siempre actualizar
    BASE = f'{REPO}/visualizacion_datos'
else:
    BASE = '../..'

DATA_DIR = f'{BASE}/data'
FIG_DIR  = f'{BASE}/data/figures'
os.makedirs(FIG_DIR, exist_ok=True)

# ── 4. Paleta ────────────────────────────────────────────────────
COLORES = {
    'RF':           '#DC2626',
    'SVM':          '#2563EB',
    'LR':           '#64748B',
    'ResNet':       '#7C3AED',
    'EfficientNet': '#0891B2',
    'UNet':         '#9CA3AF',
    'RedEdge':      '#7C3AED',
    'Topo':         '#D97706',
    'SAR':          '#0369A1',
    'Optico':       '#059669',
    'Literatura':   '#059669',
    'Colombia':     '#DC2626',
}

# ── 5. Cargar datos ──────────────────────────────────────────────
df = pd.read_csv(f'{DATA_DIR}/comparison_table.csv')
ch = pd.read_csv(f'{DATA_DIR}/channel_stats_by_class.csv')
with open(f'{DATA_DIR}/final_summary.json') as f:
    summary = json.load(f)

def load_f1s(fname, key='best_f1'):
    with open(f'{DATA_DIR}/folds/{fname}') as f: d = json.load(f)
    return [fold[key] for fold in d['folds']]

rf_folds     = load_f1s('random_forest_folds.json')
svm_folds    = load_f1s('svm_folds.json')
lr_folds     = load_f1s('logistic_regression_folds.json')
resnet_folds = load_f1s('resnet50_folds.json', key='f1_thr05')
unet_folds   = load_f1s('unet_folds.json',     key='f1_pixel_thr05')

print(f"Entorno: {'Colab' if IN_COLAB else 'Local'}")
print(f"Datos: {df.shape[0]} modelos | DATA_DIR: {DATA_DIR}")
print(df[['Modelo','F1 medio','Std']].to_string(index=False))


---
## A1 — ¿Los resultados de la literatura son comparables con los nuestros?

**Decisión:** Para saber si un modelo publicado es mejor o peor que el nuestro, primero hay que verificar que se evaluaron de la misma manera. Si no, la comparación no tiene validez.

Este proyecto usa **dos protocolos** de evaluación:
- **Protocolo HOG + DEM + NDVI** (características básicas, n=3.799) — comparable con estudios clásicos de la literatura
- **Protocolo 14 bandas** (señal completa Sentinel, n=1.500) — optimizado para el dataset Landslide4Sense

**Implicación para Colombia:** Si alguien reporta F1=0.90 en un paper pero usó validación aleatoria en lugar de geoespacial, ese número no es replicable en campo.


In [ ]:
import numpy as np

modelos_a1 = ['LR', 'SVM', 'RF']
f1_opt  = [0.7886, 0.7974, 0.8368]   # 14 bandas Sentinel
f1_base = [0.7512, 0.7340, 0.7891]   # HOG+DEM+NDVI
f1_dl   = {'ResNet-50': 0.7840, 'EfficientNet': 0.7554, 'U-Net': 0.4443}

COLOR_OPT  = '#1E3A5F'   # azul oscuro — 14 bandas
COLOR_BASE = '#93C5FD'   # azul claro  — HOG+DEM+NDVI
COLOR_DL   = '#C4B5FD'   # morado claro — Deep Learning

x = np.arange(len(modelos_a1))
ancho = 0.32

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.yaxis.grid(True, color='#E5E7EB', linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

# Barras DL
dl_nombres = list(f1_dl.keys())
dl_vals    = list(f1_dl.values())
x_dl = np.arange(len(modelos_a1), len(modelos_a1) + len(dl_nombres))
ax.bar(x_dl, dl_vals, width=ancho*1.9, color=COLOR_DL, zorder=3)
for xi, val in zip(x_dl, dl_vals):
    ax.text(xi, val+0.008, f'{val:.3f}', ha='center', fontsize=9.5, color='#6B7280')

# Barras clásicos — dos protocolos
b1 = ax.bar(x - ancho/2, f1_opt,  width=ancho, color=COLOR_OPT,  zorder=4)
b2 = ax.bar(x + ancho/2, f1_base, width=ancho, color=COLOR_BASE, zorder=4)

for bar, val in zip(b1, f1_opt):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.008, f'{val:.3f}',
            ha='center', fontsize=9.5, color='#1E3A5F', fontweight='bold')
for bar, val in zip(b2, f1_base):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.008, f'{val:.3f}',
            ha='center', fontsize=9.5, color='#374151')

# Flecha Δ en RF
rf_i = 2
ax.annotate('', xy=(rf_i+ancho/2, f1_base[rf_i]+0.004),
            xytext=(rf_i-ancho/2, f1_opt[rf_i]+0.004),
            arrowprops=dict(arrowstyle='<->', color='#DC2626', lw=2))
ax.text(rf_i+0.04, (f1_opt[rf_i]+f1_base[rf_i])/2+0.03,
        f'Δ = {f1_opt[rf_i]-f1_base[rf_i]:.3f}',
        fontsize=9, color='#DC2626', fontweight='bold')

ax.axhline(0.80, color='#6B7280', lw=1.2, ls='--', zorder=2)
ax.text(len(modelos_a1)+len(dl_nombres)-0.3, 0.803, 'F1 = 0.80', fontsize=8.5, color='#6B7280')

# Separador clásico / DL
ax.axvline(len(modelos_a1)-0.5, color='#E5E7EB', lw=1.5)
ax.text(len(modelos_a1)/2-0.2,   0.935, 'Modelos clásicos', fontsize=8.5, color='#9CA3AF', ha='center')
ax.text(len(modelos_a1)+len(dl_nombres)/2-0.3, 0.935, 'Deep Learning', fontsize=8.5, color='#9CA3AF', ha='center')

ax.set_xticks(list(x)+list(x_dl))
ax.set_xticklabels(modelos_a1+dl_nombres)
ax.set_ylim(0.35, 0.96)
ax.set_xlabel('Modelo', fontsize=10)
ax.set_ylabel('F1-Score', fontsize=10)
ax.set_title('A1 — El protocolo de evaluación cambia el resultado\n¿Los resultados de la literatura son comparables con los nuestros?',
             fontsize=11, pad=12)

leyenda = [
    mpatches.Patch(color=COLOR_OPT,  label='14 bandas del satélite (Sentinel completo)'),
    mpatches.Patch(color=COLOR_BASE, label='Características básicas del terreno (HOG+DEM+NDVI)'),
    mpatches.Patch(color=COLOR_DL,   label='Deep Learning (un solo protocolo)'),
]
ax.legend(handles=leyenda, loc='upper right', fontsize=9, frameon=True,
          framealpha=0.95, edgecolor='#E5E7EB', bbox_to_anchor=(0.99, 0.99))
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/nb02_a1_protocolo.png', dpi=150, bbox_inches='tight')
plt.show()


---
## A2 — ¿Qué tan grande es la brecha si aplicamos estos modelos en Colombia?

**Decisión:** Antes de elegir un modelo para Colombia hay que saber qué rendimiento esperar — no el que reporta el paper en su propio territorio, sino el que se obtendría al aplicarlo en Colombia sin reentrenamiento.

Los mejores modelos de la literatura (entrenados y evaluados en Nepal, Italia, Perú) alcanzan F1 entre 0.82 y 0.91. Aplicados en Colombia, la brecha de dominio (diferente geología, vegetación, cobertura nubosa) reduce ese rendimiento. El análisis LORO (*Leave-One-Region-Out*) del proyecto estima esa reducción.

**Implicación:** Hay brecha, pero no es insalvable. Con las bandas correctas (RedEdge) se puede acercar el rendimiento al estado del arte sin tener datos colombianos.


In [ ]:
import numpy as np

datos_pr = {
    'LR':        (0.7971, 0.7806),
    'SVM':       (0.8193, 0.7777),
    'RF':        (0.7439, 0.9569),
    'ResNet-50': (0.7219, 0.8771),
}
literatura = [
    ('Ghorbanzadeh et al. (2022)',         0.717, '#FECACA'),
    ('Lv et al. — L4S Competition (2022)', 0.739, '#F87171'),
    ('Liu et al. — Multi-scale (2024)',    0.760, '#EF4444'),
    ('Enhanced U-Net++ (2025)',            0.841, '#B91C1C'),
]

fig, ax = plt.subplots(figsize=(8, 6.5))
r_arr = np.linspace(0.62, 0.999, 400)

# Curvas ISO-F1 literatura
for nombre, f1_val, cl in literatura:
    p_arr = f1_val * r_arr / (2*r_arr - f1_val)
    mask  = (p_arr > 0) & (p_arr <= 1.0)
    ax.plot(r_arr[mask], p_arr[mask], color=cl, lw=1.3, ls='--', zorder=2)
    valid_r, valid_p = r_arr[mask], p_arr[mask]
    if len(valid_r):
        ax.text(valid_r[-1]-0.005, valid_p[-1]+0.010, nombre,
                fontsize=7.5, color=cl, ha='right', va='bottom')

# Puntos modelos del estudio
col_mod    = {'LR':'#6B7280','SVM':'#6B7280','RF':'#DC2626','ResNet-50':'#374151'}
marker_mod = {'LR':'o','SVM':'o','RF':'o','ResNet-50':'D'}
offsets    = {'LR':(-0.016,0.013),'SVM':(0.006,0.013),
              'RF':(0.006,-0.024),'ResNet-50':(0.006,0.013)}
for nm, (prec, rec) in datos_pr.items():
    ax.scatter(rec, prec, s=130, color=col_mod[nm], zorder=6,
               edgecolors='white', linewidth=1.5, marker=marker_mod[nm])
    dx, dy = offsets[nm]
    ax.text(rec+dx, prec+dy, nm, fontsize=9.5, color=col_mod[nm],
            fontweight='bold' if nm=='RF' else 'normal')

# Anotación RF
ax.annotate('RF — mayor cobertura,\nalcanza benchmark 2025',
            xy=(0.9569, 0.7439), xytext=(0.785, 0.865),
            arrowprops=dict(arrowstyle='->', color='#DC2626', lw=1.5),
            fontsize=8.5, color='#DC2626',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor='#DC2626', alpha=0.92))

ax.set_xlim(0.62, 1.02)
ax.set_ylim(0.58, 0.90)
ax.set_xlabel('Cobertura (Recall) — fracción de deslizamientos reales detectados', fontsize=10)
ax.set_ylabel('Precisión — de las alertas, ¿cuántas son correctas?', fontsize=10)
ax.set_title('A2 — ¿Qué tan grande es la brecha frente a la literatura internacional?\n'
             'Curvas punteadas = F1 reportado en estudios publicados',
             fontsize=11, pad=10)

leyenda = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#6B7280', markersize=10,
           label='Modelos clásicos (Landslide4Sense)'),
    Line2D([0],[0], marker='D', color='w', markerfacecolor='#374151', markersize=9,
           label='Deep Learning (Landslide4Sense)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#DC2626', markersize=10,
           label='RF — mejor resultado'),
    Line2D([0],[0], color='#EF4444', ls='--', lw=1.5,
           label='Benchmarks internacionales (ISO-F1)'),
]
ax.legend(handles=leyenda, loc='upper left', fontsize=8.5,
          frameon=True, framealpha=0.95, edgecolor='#E5E7EB')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/nb02_a2_brecha_colombia.png', dpi=150, bbox_inches='tight')
plt.show()


---
## A3 — ¿Vale la pena invertir en arquitectura compleja para Colombia?

**Decisión:** Colombia tiene recursos computacionales y de infraestructura limitados. ¿Justifica el costo de entrenar una red neuronal profunda cuando un modelo más simple da el mismo o mejor resultado?

**Implicación:** No. Los datos muestran que complejidad y rendimiento no escalan juntos en este problema. Un modelo clásico es más rápido, más interpretable y más fácil de mantener — ventajas críticas para un organismo público de gestión de riesgos.


In [ ]:
modelos_a3  = ['U-Net\nResNet34', 'EfficientNet', 'ResNet-50', 'LR', 'SVM', 'RF']
f1_a3       = [0.4443, 0.7554, 0.7840, 0.7886, 0.7974, 0.8368]
complejidad = ['Alta', 'Alta', 'Alta', 'Baja', 'Media', 'Media']
col_a3      = ['#C4B5FD','#C4B5FD','#C4B5FD','#9CA3AF','#9CA3AF','#DC2626']

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.xaxis.grid(True, color='#E5E7EB', linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

bars = ax.barh(modelos_a3, f1_a3, color=col_a3, height=0.55, zorder=3)

for bar, comp in zip(bars, complejidad):
    ax.text(0.01, bar.get_y()+bar.get_height()/2,
            f'Complejidad: {comp}', va='center', fontsize=8.5,
            color='white', fontweight='bold')

for bar, val in zip(bars, f1_a3):
    ax.text(val+0.005, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10, color='#374151')

ax.axvline(0.80, color='#6B7280', lw=1.2, ls='--', zorder=2)
ax.text(0.801, 5.65, 'F1 = 0.80', fontsize=8.5, color='#6B7280')

ax.set_xlim(0, 0.97)
ax.set_xlabel('F1-Score — capacidad de detección', fontsize=10)
ax.set_ylabel('Modelo', fontsize=10)
ax.set_title('A3 — ¿Vale la pena invertir en arquitectura compleja para Colombia?\n'
             'Complejidad del modelo vs. resultado real',
             fontsize=11, pad=12)

leyenda = [
    mpatches.Patch(color='#C4B5FD', label='Deep Learning'),
    mpatches.Patch(color='#9CA3AF', label='Modelos clásicos'),
    mpatches.Patch(color='#DC2626', label='Mejor resultado (RF)'),
]
ax.legend(handles=leyenda, loc='lower right', fontsize=9, frameon=False)
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/nb02_a3_complejidad.png', dpi=150, bbox_inches='tight')
plt.show()


---
## A4 — ¿Qué modelo usar como punto de partida en Colombia?

**Decisión:** Integrar los hallazgos anteriores en un único argumento de selección. No se trata solo del modelo con mayor F1 en un dataset extranjero — se trata del modelo más adecuado dado el contexto colombiano: sin datos propios, con recursos limitados y necesidad de transparencia.

Cuatro criterios evaluados simultáneamente por modelo:
- **F1-Score** — rendimiento en Landslide4Sense (dato empírico)
- **Consistencia** — estabilidad entre folds (menor varianza = más predecible)
- **Costo computacional bajo** — viabilidad con infraestructura local
- **Interpretabilidad** — capacidad de explicar el resultado a tomadores de decisión


In [ ]:
import numpy as np

modelos   = ['RF', 'SVM', 'LR', 'ResNet-50', 'EfficientNet', 'U-Net']
criterios = ['F1-Score', 'Consistencia', 'Costo bajo', 'Interpretabilidad']

# Scores 0-1 por criterio — mayor = mejor
scores = {
    'RF':          [0.837, 0.95, 0.90, 0.90],
    'SVM':         [0.797, 0.70, 0.85, 0.70],
    'LR':          [0.789, 0.81, 0.95, 0.80],
    'ResNet-50':   [0.784, 0.85, 0.30, 0.25],
    'EfficientNet':[0.755, 0.82, 0.25, 0.20],
    'U-Net':       [0.444, 0.78, 0.20, 0.15],
}

# Colores por criterio: más crítico = mayor contraste
col_criterios = ['#1E3A5F', '#2563EB', '#93C5FD', '#BFDBFE']

n_mod  = len(modelos)
n_crit = len(criterios)
w      = 0.18
x      = np.arange(n_mod)
offsets = np.linspace(-(n_crit-1)/2*w, (n_crit-1)/2*w, n_crit)

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.yaxis.grid(True, color='#E5E7EB', linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

for j, (crit, col, off) in enumerate(zip(criterios, col_criterios, offsets)):
    vals = [scores[m][j] for m in modelos]
    bars = ax.bar(x + off, vals, width=w*0.9, color=col, zorder=3,
                  label=crit, alpha=0.92)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.015,
                f'{val:.2f}', ha='center', fontsize=7,
                color='#374151')

# Separador Clásicos / DL
sep = 2.5  # entre LR (idx 2) y ResNet (idx 3)
ax.axvline(sep, color='#6B7280', lw=1.5, ls='--', zorder=4)
ax.text(1.0,  1.07, 'Modelos clásicos', ha='center', fontsize=9.5,
        color='#374151', fontweight='bold')
ax.text(4.0,  1.07, 'Deep Learning',    ha='center', fontsize=9.5,
        color='#374151', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(modelos, fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_xlabel('Modelo', fontsize=11)
ax.set_ylabel('Puntuación (0 = peor · 1 = mejor)', fontsize=11)
ax.set_title('A4 — Perfil de decisión por modelo\n'
             '¿Cuál usar como punto de partida en Colombia sin datos locales?',
             fontsize=11, pad=12)
ax.legend(loc='upper right', fontsize=9, frameon=True,
          framealpha=0.95, edgecolor='#E5E7EB', title='Criterio', title_fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/nb02_a4_scorecard.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Conclusión — Respuesta a la pregunta central

> *¿Cuál modelo debería usarse como punto de partida para una herramienta de alerta en Colombia?*

**Random Forest**, entrenado con datos internacionales (Landslide4Sense).

| Criterio | Resultado | Implicación |
|----------|-----------|-------------|
| **Protocolo de evaluación** | Usar 5-fold geoespacial | Comparaciones válidas con literatura |
| **Brecha con el estado del arte** | Alcanza benchmark 2025 en cobertura | Brecha real pero no insalvable |
| **Costo computacional** | Modelos clásicos ≈ Deep Learning en F1 | No justifica la complejidad |
| **Perfil de decisión** | RF lidera en F1, consistencia e interpretabilidad | Mejor candidato para Colombia |

> **Lo que falta:** un dataset etiquetado colombiano.  
> Con él, Random Forest es el punto de partida más rápido de adaptar y validar localmente.
